# 1/2 - Train the denoiser (Kaggle or Colab GPU)

Runs on either platform. The setup cell finds the code, data and any
checkpoint wherever they landed - Kaggle mounts datasets at `/kaggle/input`,
Colab pulls them into `/root/.cache/kagglehub`.

Training and benchmarking are separate notebooks because together they exceed
Kaggle's 12 h limit.

## Before running

**Kaggle:** *Add Input* -> `pointdenoise-code`, `pointdenoise-data`, and your
checkpoint dataset. *Settings* -> *Accelerator* -> **GPU T4 x2**.

**Colab:** *Runtime* -> *Change runtime type* -> **T4 GPU**, then run the
kagglehub cell that downloads the datasets before this one.

## Resuming

If a checkpoint is found the run continues from its epoch rather than starting
over. From epoch 46 that is ~2 h instead of ~8 h.

## After

Download `best.pt` and `history.json`, upload `best.pt` as a dataset, then run
the benchmark notebook against it.


In [ ]:
import glob, os, subprocess, sys

# Works on Kaggle and on Colab. Kaggle mounts datasets at /kaggle/input; Colab
# pulls them with kagglehub into /root/.cache/kagglehub. Search both rather
# than assuming, so the same notebook runs either place.
SEARCH_ROOTS = [
    "/kaggle/input",
    "/root/.cache/kagglehub",
    "/content",
    os.getcwd(),
]
SEARCH_ROOTS = [r for r in SEARCH_ROOTS if os.path.isdir(r)]

def show_tree(root, limit=25):
    n = 0
    for base, dirs, files in os.walk(root):
        depth = base.replace(root, "").count(os.sep)
        if depth > 3:
            continue
        print("  " * depth + os.path.basename(base) + "/")
        for f in files[:2]:
            print("  " * (depth + 1) + f)
        if len(files) > 2:
            print("  " * (depth + 1) + "... (%d files)" % len(files))
        n += 1
        if n > limit:
            print("  ...truncated")
            return

def find_containing(*markers):
    """First directory under any search root that holds one of `markers`."""
    for root in SEARCH_ROOTS:
        for base, dirs, files in os.walk(root):
            for m in markers:
                if m in dirs or m in files:
                    return base
    return None

CODE = find_containing("pointdenoise")
DATA = find_containing("examples", "PUNet", "PCNet")
CKPT = None
for root in SEARCH_ROOTS:
    for base, _, files in os.walk(root):
        for f in ("best.pt", "last.pt"):
            if f in files:
                CKPT = os.path.join(base, f)
                break
        if CKPT:
            break
    if CKPT:
        break

print("roots :", SEARCH_ROOTS)
print("code  :", CODE)
print("data  :", DATA)
print("ckpt  :", CKPT or "(none - training will start from scratch)")

if not CODE or not DATA:
    for root in SEARCH_ROOTS:
        print()
        print("=== " + root + " ===")
        show_tree(root)
    missing = "pointdenoise-code" if not CODE else "pointdenoise-data"
    raise SystemExit(
        "\nCould not find the " + missing + " dataset.\n"
        "On Kaggle: use 'Add Input' and attach it.\n"
        "On Colab: run kagglehub.dataset_download('<user>/" + missing + "') first.\n"
        "The trees above show what is actually present."
    )

sys.path.insert(0, CODE)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "trimesh", "rtree"], check=False)

# Colab and Kaggle both give a T4, but Colab needs the runtime type set.
import torch
print()
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NO GPU - Colab: Runtime > Change runtime type > T4. Kaggle: Settings > Accelerator")

OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content/out"
os.makedirs(OUT, exist_ok=True)
print("outputs ->", OUT)


## Noise is sampled per patch, not fixed

Run 1 trained at a fixed 2% and scored +56% CD at 2%, +73% at 3%, and only
+2% at 1%: it had learned one correction size and applied it regardless, so at
low noise it moved points that were already close. On a sphere the same model
came out 84% *worse* than the noisy input at 1%. Sampling the range the
benchmark actually tests fixes that and is better at every level.


In [ ]:
from pointdenoise.benchmark import load_training_clouds
from pointdenoise.data import Shape
import numpy as np

train_clouds = load_training_clouds(DATA, "PUNet", "sparse")
shapes = [Shape(pts, noise_level=0.02, rng=np.random.default_rng(i))
          for i, (_, pts) in enumerate(train_clouds)]
print(f"{len(shapes)} training shapes, {shapes[0].clean.shape[0]} points each")


In [ ]:
from pointdenoise.engine import train

# CKPT was located by the setup cell. train() restores model, optimizer,
# scheduler and epoch counter, so this continues where the last run stopped
# rather than starting over.
EPOCHS = 60

if CKPT:
    import torch
    ck = torch.load(CKPT, map_location="cpu", weights_only=False)
    done = ck["epoch"]
    print(f"resuming from {CKPT}")
    print(f"  at epoch {done}/{EPOCHS}, best loss {ck.get('best'):.6f}")
    print(f"  {EPOCHS - done} epochs left, roughly {(EPOCHS - done) * 8 / 60:.1f} h")
else:
    print(f"no checkpoint found - training {EPOCHS} epochs from scratch (~8 h)")

model, history = train(
    shapes,
    out_dir=OUT + "/runs",
    epochs=EPOCHS,
    batch_size=32,
    points_per_patch=256,
    patches_per_shape=1000,
    lr=1e-3,
    repulsion_weight=0.05,
    noise_range=(0.005, 0.03),
    model_kwargs={"d_model": 256, "num_heads": 8, "num_layers": 6},
    num_workers=2,
    seed=0,
    resume=CKPT,
)


In [ ]:
import matplotlib.pyplot as plt, shutil

fig, (a, b) = plt.subplots(1, 2, figsize=(13, 4))
a.plot([h["total"] for h in history], label="total")
a.plot([h["chamfer"] for h in history], label="chamfer")
a.set_xlabel("epoch"); a.set_ylabel("loss"); a.legend(); a.grid(alpha=.3)
a.set_title("Training loss")
b.plot([h["lr"] for h in history], color="tab:orange")
b.set_yscale("log"); b.set_xlabel("epoch"); b.set_ylabel("lr"); b.grid(alpha=.3)
b.set_title("Learning rate")
plt.tight_layout(); plt.savefig(OUT + "/loss.png", dpi=120); plt.show()

print(f"epoch {history[0]['epoch']} {history[0]['total']:.6f} -> "
      f"epoch {history[-1]['epoch']} {history[-1]['total']:.6f}")
print(f"total {sum(h['seconds'] for h in history)/3600:.1f} h")
shutil.copy(OUT + "/runs/best.pt", OUT + "/best.pt")
print("\ndownload best.pt and history.json, then run the benchmark notebook")
